In [4]:
from roboflow import Roboflow
import os

API_KEY = "CfVGS82wt63mHysL62v4"
DATASET_PATH = r"d:\Dicoding\capstone-project\SIDIAS-\data_science\data\processed\gambar"


# Buat folder baru
os.makedirs(DATASET_PATH, exist_ok=True)

# Download dengan overwrite
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("mnt-bgmps").project("deteksi-stunting-jj1oq")
dataset = project.version(1).download("coco", location=DATASET_PATH, overwrite=True)

print(f"\n✓ Download selesai!")
print(f"Dataset ada di: {DATASET_PATH}")

# Verifikasi
print(f"\n=== Verifikasi ===")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root) or 'dataset_gambar'
    print(f"{indent}📁 {folder}/")
    
    subindent = '  ' * (level + 1)
    for f in files[:5]:
        print(f"{subindent}📄 {f}")
    if len(files) > 5:
        print(f"{subindent}... dan {len(files)-5} file lainnya")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to d:\Dicoding\capstone-project\SIDIAS-\data_science\data\processed\gambar in coco:: 100%|██████████| 1861/1861 [00:00<00:00, 3137.66it/s]


✓ Download selesai!
Dataset ada di: d:\Dicoding\capstone-project\SIDIAS-\data_science\data\processed\gambar

=== Verifikasi ===
📁 gambar/
  📄 README.dataset.txt
  📄 README.roboflow.txt
  📁 test/
    📄 Malnurished-135-_jpg.rf.a25639f5b696a1dd5cfff1d4be39e4bb.jpg
    📄 Malnurished-14-_jpg.rf.986b73314b295d7094eb1f89a00668b2.jpg
    📄 Malnurished-152-_jpg.rf.d29c4897af9102370263a955c698e490.jpg
    📄 Malnurished-153-_jpg.rf.59523cabdb710a7a222a713e32df0f3e.jpg
    📄 Malnurished-154-_jpg.rf.546e431b5fef76d3cd2844ddb45f1d99.jpg
    ... dan 182 file lainnya
  📁 train/
    📄 Malnurished-1-_jpeg.rf.cb8004bb25bfc6205b785ef4036b98dc.jpg
    📄 Malnurished-1-_jpg.rf.739a30fba704bc319a95d1835316b149.jpg
    📄 Malnurished-1-_png.rf.01b94a181d72cd8e0521cf0d9b807e2b.jpg
    📄 Malnurished-1-_webp.rf.f16baef35ae422da18658e55775f69ae.jpg
    📄 Malnurished-10-_jpeg.rf.b605fcd0385d58786e4973f6116b6592.jpg
    ... dan 1294 file lainnya
  📁 valid/
    📄 Malnurished-10-_jpg.rf.9521b02ff15a8580ed7ca6397137773

In [5]:
import os
import json
import shutil
from tqdm import tqdm

def process_clean_and_purge(target_folder):
    folder_path = os.path.join(DATASET_PATH, target_folder)
    
    if not os.path.exists(folder_path):
        print(f"⚠️ Folder {target_folder} tidak ditemukan.")
        return

    # 1. Cari file JSON
    json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    if not json_files:
        print(f"⚠️ Tidak ada file JSON di {target_folder}, mungkin sudah diproses.")
    else:
        json_path = os.path.join(folder_path, json_files[0])
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        categories = {cat['id']: cat['name'] for cat in data['categories']}
        images = {img['id']: img['file_name'] for img in data['images']}

        # 2. Buat sub-folder label
        for cat_name in categories.values():
            os.makedirs(os.path.join(folder_path, cat_name), exist_ok=True)

        # 3. Proses Pindah (Labeling)
        print(f"📦 Mengatur folder {target_folder}...")
        for ann in tqdm(data['annotations'], desc=f"Moving {target_folder}"):
            img_name = images.get(ann['image_id'])
            cat_name = categories.get(ann['category_id'])
            
            src = os.path.join(folder_path, img_name)
            dst = os.path.join(folder_path, cat_name, img_name)
            
            if os.path.exists(src):
                shutil.move(src, dst)

    # 4. SAPU BERSIH FILE & FOLDER KOSONG (Revisi Tim AI)
    all_items = os.listdir(folder_path)
    for item in all_items:
        item_path = os.path.join(folder_path, item)
        
        # Hapus semua FILE sisa (JSON, TXT, dll)
        if os.path.isfile(item_path):
            os.remove(item_path)
            
        # Hapus FOLDER jika kosong atau jika namanya 'Deteksi-Stunting'
        elif os.path.isdir(item_path):
            # Jika folder kosong atau namanya mengandung 'Deteksi-Stunting'
            if not os.listdir(item_path) or 'Deteksi-Stunting' in item:
                shutil.rmtree(item_path)
                print(f"🗑️ Folder sampah dihapus: {item}")

    print(f"✨ Folder {target_folder} SELESAI: Bersih & Minimalis.")

# Eksekusi
for f in ['train', 'valid', 'test']:
    process_clean_and_purge(f)

# 5. Laporan Akhir
print("\n=== LAPORAN FINAL DATA SCIENCE ===")
for f in ['train', 'valid', 'test']:
    p = os.path.join(DATASET_PATH, f)
    if os.path.exists(p):
        print(f"\n📂 {f.upper()}: {os.listdir(p)}")

📦 Mengatur folder train...


Moving train:   0%|          | 0/1175 [00:00<?, ?it/s]

Moving train: 100%|██████████| 1175/1175 [00:00<00:00, 2877.59it/s]


🗑️ Folder sampah dihapus: Deteksi-Stunting
✨ Folder train SELESAI: Bersih & Minimalis.
📦 Mengatur folder valid...


Moving valid: 100%|██████████| 334/334 [00:00<00:00, 3789.42it/s]


🗑️ Folder sampah dihapus: Deteksi-Stunting
✨ Folder valid SELESAI: Bersih & Minimalis.
📦 Mengatur folder test...


Moving test: 100%|██████████| 171/171 [00:00<00:00, 4119.74it/s]

🗑️ Folder sampah dihapus: Deteksi-Stunting
✨ Folder test SELESAI: Bersih & Minimalis.

=== LAPORAN FINAL DATA SCIENCE ===

📂 TRAIN: ['Healthy', 'MalNutrisi', 'Stunting']

📂 VALID: ['Healthy', 'MalNutrisi', 'Stunting']

📂 TEST: ['Healthy', 'MalNutrisi', 'Stunting']


In [6]:
import albumentations as A
import cv2
import os
import glob
from tqdm import tqdm

# 1. Konfigurasi
TARGET_COUNT = 550
folder_to_augment = os.path.join(DATASET_PATH, "train", "MalNutrisi")

# 2. Definisi Teknik Augmentasi (Sederhana tapi Ampuh)
# Kita buat variasi posisi dan cahaya agar model AI tidak bosan
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
])

# 3. Ambil data asli yang cuma 24 biji itu
current_files = glob.glob(os.path.join(folder_to_augment, "*.jpg")) + \
                glob.glob(os.path.join(folder_to_augment, "*.png"))

num_existing = len(current_files)
num_needed = TARGET_COUNT - num_existing

if num_needed > 0:
    print(f"🔥 Menambah {num_needed} gambar augmentasi untuk MalNutrisi...")
    
    generated_count = 0
    pbar = tqdm(total=num_needed)
    
    while generated_count < num_needed:
        for img_path in current_files:
            if generated_count >= num_needed: break
            
            # Baca gambar
            image = cv2.imread(img_path)
            if image is None: continue
            
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Eksekusi Augmentasi
            augmented = transform(image=image)['image']
            
            # Simpan hasil
            augmented_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
            new_name = f"aug_{generated_count}.jpg"
            cv2.imwrite(os.path.join(folder_to_augment, new_name), augmented_bgr)
            
            generated_count += 1
            pbar.update(1)
    pbar.close()

print(f"✅ Sekarang total gambar MalNutrisi: {len(os.listdir(folder_to_augment))}")

ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.


🔥 Menambah 526 gambar augmentasi untuk MalNutrisi...


100%|██████████| 526/526 [00:00<00:00, 953.98it/s] 

✅ Sekarang total gambar MalNutrisi: 550


In [7]:
import os

# Cek folder train saja untuk verifikasi akhir
train_path = os.path.join(DATASET_PATH, "train")
print("=== FINAL DATASET REPORT (TRAIN) ===")

total_data = 0
for label in os.listdir(train_path):
    label_dir = os.path.join(train_path, label)
    if os.path.isdir(label_dir):
        count = len(os.listdir(label_dir))
        print(f"✅ {label.ljust(15)}: {count} gambar")
        total_data += count

print("-" * 30)
print(f"TOTAL DATA TRAINING: {total_data} gambar")

=== FINAL DATASET REPORT (TRAIN) ===
✅ Healthy        : 614 gambar
✅ MalNutrisi     : 550 gambar
✅ Stunting       : 531 gambar
------------------------------
TOTAL DATA TRAINING: 1695 gambar
